In [1]:
%load_ext autoreload
%autoreload 2

import os
import glob
from pathlib import Path
import sys
import pyarrow
import pandas as pd


PROJECT_ROOT = Path().resolve().parent
sys.path.append(str(PROJECT_ROOT))

from src.config.config import DATA_PROCESSED, DATA_FEATURES, MODELS_ANOMALYD
from src.loaders.csv_loader import CSVLoader
from src.repository.parquet_repository import ParquetRepository

from src.models.IsolationForest import IsolationForestAnalyzer


csvLoader = CSVLoader()

In [2]:
repo = ParquetRepository(DATA_FEATURES)

In [3]:
df_load = repo.load("beverage_sales_feature.parquet")

[OK] Arquivo carregado: D:\PROJETOS\git_repo\BEVERAGE-SALES\data\features\beverage_sales_feature.parquet


In [4]:
print (df_load.dtypes)

Order_Date                  datetime64[ns]
Category                            object
Product                             object
Region                              object
quantity_sum                       float64
total_price_sum                    float64
unit_price_mean                    float64
discount_mean                      float64
order_count                        float64
customer_count                     float64
avg_ticket                         float64
day_of_week                          int32
month                                int32
year                                 int32
is_weekend                           int64
quantity_sum_mean_7d               float64
quantity_sum_std_7d                float64
quantity_sum_sum_7d                float64
total_price_sum_mean_7d            float64
total_price_sum_std_7d             float64
total_price_sum_mean_30d           float64
unit_price_mean_mean_7d            float64
discount_mean_mean_14d             float64
quantity_vs

# Atencao
Para evitar Data Leakage, o modelo Isolation Forest é treinado apenas para o periodo de 2021 a 2022 deixando o ano 2023 para testes

In [7]:
df_load["Order_Date"] = pd.to_datetime(df_load["Order_Date"])

df_train_if = df_load[df_load["Order_Date"].dt.year.isin([2021, 2022])].copy()
df_test_if = df_load[df_load["Order_Date"].dt.year == 2023].copy()

In [8]:
if_analyzer = IsolationForestAnalyzer(
    random_state=42,
    cv=3,
    n_jobs=-1,
    verbose=1
)

if_analyzer.fit(df_train_if)



Fitting 3 folds for each of 72 candidates, totalling 216 fits


In [9]:
print(if_analyzer.get_best_params())


{'model__contamination': 0.01, 'model__max_features': 1.0, 'model__max_samples': 512, 'model__n_estimators': 100}


In [10]:
model_path = if_analyzer.save_model(
    folder_path=MODELS_ANOMALYD,
    file_name="isolation_forest_analyzer.joblib"
)

In [11]:
model_path = if_analyzer.save_best_params_json(
    folder_path=MODELS_ANOMALYD,
    file_name="isolation_forest_best_params.json"
)

In [ ]:
df_if = if_analyzer.predict(df_load)

cols_anomaly_output = [
    "Order_Date",
    "Product",
    "Region",
    "anomaly_flag",
    "anomaly_label",
    "anomaly_score",
    "if_qty_signal",
    "if_sales_signal",
    "if_discount_signal",
    "if_ticket_signal"
]

df_if_output = df_if[cols_anomaly_output].copy()



In [ ]:
summary_if = if_analyzer.anomaly_summary(df_if)
print(summary_if.head(20))

In [ ]:
repo.set_base_path(DATA_FEATURES)
repo.save(df_if, "anomaly_predictions.parquet")